<a href="https://colab.research.google.com/github/Imposon/Trial/blob/main/Copy_of_LangChain_basic_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip uninstall -y langgraph-prebuilt langgraph
!pip install langchain==0.2.14 langchain-core==0.2.43 langchain-groq==0.1.5 -q

In [ ]:
# !pip uninstall -y langgraph-prebuilt langgraph

In [ ]:
# !pip install -q --upgrade langchain langchain-groq

In [3]:
import os
from google.colab import userdata
from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq

# Retrieve the secret value as a string
os.environ["GROQ_API_KEY"] = userdata.get('OPOP')


In [7]:
# Test if your api key works

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")

print(llm.invoke("who am i"))

content="Unfortunately, I'm a large language model, I don't have any information about your personal identity or details. Our conversation just started, and I don't have any context about you.\n\nHowever, I can try to play a game with you to help you discover more about yourself or your interests. If you'd like, we can engage in a fun conversation, and I can ask you some questions to help you reflect on who you are and what you're passionate about.\n\nSo, would you like to play a game, or is there something specific you'd like to talk about or ask me? I'm here to help and chat with you!" response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 38, 'total_tokens': 168, 'completion_time': 0.450512228, 'completion_tokens_details': None, 'prompt_time': 0.00107715, 'prompt_tokens_details': None, 'queue_time': 0.159492965, 'total_time': 0.451589378}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'finish_reason': 'stop', 'logprobs': None}

# LangChain Exercise: Manual LLM Chaining with ChatGroq

## Objective

In this exercise, you will build a **multi-step LLM pipeline** using LangChain — without relying on high-level chain abstractions.

You will:
1. Generate an explanation of a topic  
2. Use that explanation to create quiz questions  

This demonstrates how **outputs from one LLM call can be used as inputs to another**.

Instead of using `LLMChain` or `SequentialChain`, you will implement this flow **manually** using `.invoke()`.

---

## Key Components Used

### 1. `ChatGroq`
- Interface to large language models via Groq
- Extremely fast inference
- Used to generate responses from prompts

### 2. `PromptTemplate`
- Used to create **structured and reusable prompts**
- Allows you to insert variables dynamically into prompts
- Helps maintain clean and consistent prompt design


### 3. `format()`
- Fills in the variables inside a PromptTemplate
- Converts the template into a final prompt string that can be sent to the LLM

### 4. `invoke()`
- Sends the formatted prompt to the LLM
- Executes the model and returns a response object

### 5. `content()`
- Extracts the actual text output from the response object
- This is what you use for the next step in the pipeline


`Input` → `Prompt` → `LLM` → `Output` → `Next Prompt` → `LLM` → `Final Output`

In [9]:
from langchain.prompts.prompt import Prompt
from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq

# Initialize the ChatGroq model (LLM instance)
# parameterize model name llama-3.3-70b-versatile and temperature 0.7 (controls randomness of output)
llm = ChatGroq(model = "llama-3.3-70b-versatile", temperature= 0.7 )


# Step 1: Explanation prompt
# Initialize prompt1 object by instantiating PromptTemplate class
# parameterize input_variables list with "topic"
# parameterize template string with "Explain {topic} in simple terms"

prompt1 = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)



# Step 2: Quiz prompt
# Initialize prompt2 object by instantiating PromptTemplate class
# parameterize input_variables list with "explanation"
# parameterize template string with a "Create 2 quiz questions from this:\n{explanation}"

prompt2 = PromptTemplate(
    input_variables=["explanation"],
    template="Create 2 quiz questions from this:\n{explanation}"
)




# ---- Execution (manual chaining) ----

# Step 1
# initialize formatted_prompt1 by calling .format() on prompt1
# parameterize format method with topic="KNN"
# invoke LLM using .invoke() with formatted prompt string
# store response object in response1
# extract generated text using response1.content and store in explanation
formatted_prompt1 = prompt1.format(topic="Programmer")
response1 = llm.invoke(formatted_prompt1)
explanation = response1.content





# Step 2
# initialize formatted_prompt2 by calling .format() on prompt2
# parameterize format method with explanation from previous step
# invoke LLM using .invoke() with formatted prompt string
# store response object in response2
# extract generated text using response2.content and store in quiz
formatted_prompt2 = prompt2.format(explanation=explanation)
response2 = llm.invoke(formatted_prompt2)
quiz = response2.content




# Final result
# aggregate outputs into a dictionary for structured access
result = {
    "explanation": explanation,
    "quiz": quiz
}

# display final output
print(result)

{'explanation': '**What is a Programmer?**\n\nA programmer, also known as a software developer or coder, is a person who creates instructions that a computer can understand. These instructions are called "code" or "programs."\n\n**What does a Programmer do?**\n\nA programmer\'s main job is to:\n\n1. **Design**: Plan and create a blueprint for a software program or application.\n2. **Write Code**: Write the instructions (code) that the computer will follow to perform a specific task.\n3. **Test**: Check the code to make sure it works correctly and fix any errors.\n4. **Maintain**: Update and improve the code to keep it running smoothly and efficiently.\n\n**Example:**\n\nImagine you want to build a house. An architect designs the blueprint, a construction worker builds the house, and a maintenance worker fixes any issues. A programmer is like all three roles combined:\n\n* They design the blueprint (software program)\n* They build the house (write the code)\n* They fix any issues (test 

In [11]:
print(result.keys())

dict_keys(['explanation', 'quiz'])


In [12]:
print(result["quiz"])

Here are 2 quiz questions based on the provided text:

1. What are the four main tasks that a programmer is responsible for?
A) Design, Write Code, Test, and Deploy
B) Design, Write Code, Test, and Maintain
C) Plan, Build, Fix, and Update
D) Create, Develop, Implement, and Manage

Answer: B) Design, Write Code, Test, and Maintain

2. What is the role of a programmer similar to, according to the example given?
A) Only an architect
B) Only a construction worker
C) A combination of an architect, a construction worker, and a maintenance worker
D) A combination of a designer, a builder, and a manager

Answer: C) A combination of an architect, a construction worker, and a maintenance worker
